# haberman: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('02_haberman_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,36.349,22.054,46.476,14.853,93.333,54.093,45.186,67.644,29.201
smote,59.215,43.792,42.243,47.390,76.000,61.695,46.089,67.656,51.504
bsmote,58.903,43.833,42.129,50.000,72.889,61.444,43.968,66.047,51.368
ros,58.847,43.705,43.857,45.074,78.222,61.648,47.993,67.761,51.276
adasyn,60.084,44.873,41.773,52.463,71.333,61.898,43.678,66.141,52.479
casb,58.859,43.898,45.181,44.228,80.000,62.114,46.351,67.521,51.379
casb_relaxed,57.699,42.095,40.958,44.926,75.556,60.241,47.396,67.199,49.897
casb_edited,62.771,47.720,42.632,56.728,71.556,64.142,49.310,69.505,55.246
casb_ros,57.142,41.990,44.157,41.213,80.444,60.829,45.845,67.739,49.566


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,37.160,21.685,30.500,17.279,84.000,50.640,37.315,63.109,29.422
smote,53.769,37.136,33.771,41.912,70.000,55.956,36.046,61.810,45.452
bsmote,56.087,39.587,34.597,46.801,68.444,57.623,35.406,61.819,47.837
ros,57.253,41.229,37.652,47.463,71.111,59.287,38.440,62.934,49.241
adasyn,59.274,43.249,37.612,51.838,68.889,60.364,36.707,62.551,51.262
casb,55.709,39.696,37.643,43.199,73.778,58.488,37.895,63.826,47.703
casb_relaxed,56.053,39.728,36.877,43.824,72.667,58.245,36.633,63.454,47.891
casb_edited,63.693,48.361,40.691,61.103,67.556,64.329,40.386,65.507,56.027
casb_ros,57.683,41.778,38.865,46.324,73.333,59.828,37.527,63.179,49.731


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,44.245,29.016,43.494,22.831,87.556,55.193,45.351,67.762,36.630
smote,52.434,36.210,36.618,36.250,78.000,57.125,42.717,68.326,44.322
bsmote,54.837,38.989,39.694,38.750,79.111,58.931,42.472,68.222,46.913
ros,56.745,40.930,39.691,42.500,76.889,59.694,41.764,66.836,48.837
adasyn,56.044,40.191,39.936,40.625,78.000,59.312,43.105,68.268,48.118
casb,53.652,37.910,40.094,36.912,80.000,58.456,43.321,67.356,45.781
casb_relaxed,56.554,41.216,41.805,41.176,79.333,60.255,45.629,69.119,48.885
casb_edited,64.761,49.591,42.945,59.853,71.111,65.482,47.964,70.543,57.176
casb_ros,53.154,37.121,38.217,36.985,78.222,57.604,44.133,67.423,45.138


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,33.525,19.991,56.000,12.279,96.000,54.140,47.481,67.038,26.758
smote,59.062,44.722,49.014,43.676,82.667,63.172,48.484,68.149,51.892
bsmote,58.912,44.075,45.619,44.926,79.778,62.352,47.905,67.333,51.493
ros,59.588,45.153,47.519,44.301,82.000,63.151,48.169,67.658,52.370
adasyn,62.424,47.960,47.180,51.140,78.000,64.570,48.938,68.600,55.192
casb,58.678,44.440,49.520,41.801,84.000,62.901,47.841,67.667,51.559
casb_relaxed,58.765,44.331,48.518,42.426,82.889,62.658,47.745,67.748,51.548
casb_edited,63.707,49.416,46.289,56.103,74.667,65.385,49.932,69.600,56.561
casb_ros,58.759,44.591,49.853,41.801,84.222,63.012,47.861,67.666,51.675


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,37.820,23.186,44.118,16.811,90.222,53.516,43.833,66.388,30.503
smote,56.120,40.465,40.411,42.307,76.667,59.487,43.334,66.485,48.292
bsmote,57.185,41.621,40.510,45.119,75.056,60.088,42.438,65.855,49.403
ros,58.108,42.754,42.180,44.835,77.056,60.945,44.092,66.297,50.431
adasyn,59.457,44.068,41.625,49.017,74.056,61.536,43.107,66.390,51.763
casb,56.724,41.486,43.109,41.535,79.444,60.490,43.852,66.593,49.105
casb_relaxed,57.268,41.842,42.039,43.088,77.611,60.350,44.350,66.880,49.555
casb_edited,63.733,48.772,43.139,58.447,71.222,64.834,46.898,68.789,56.252
casb_ros,56.684,41.370,42.773,41.581,79.056,60.318,43.841,66.501,49.027


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,62.771,adasyn,60.084,2.687,WIN
1,f1,casb_edited,47.720,adasyn,44.873,2.847,WIN
2,precision,casb,45.181,ros,43.857,1.325,WIN
3,recall,casb_edited,56.728,adasyn,52.463,4.265,WIN


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,63.693,adasyn,59.274,4.418,WIN
1,f1,casb_edited,48.361,adasyn,43.249,5.112,WIN
2,precision,casb_edited,40.691,ros,37.652,3.039,WIN
3,recall,casb_edited,61.103,adasyn,51.838,9.265,WIN


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,64.761,ros,56.745,8.015,WIN
1,f1,casb_edited,49.591,ros,40.930,8.661,WIN
2,precision,casb_edited,42.945,adasyn,39.936,3.009,WIN
3,recall,casb_edited,59.853,ros,42.500,17.353,WIN


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,63.733,adasyn,59.457,4.276,WIN
1,f1,casb_edited,48.772,adasyn,44.068,4.704,WIN
2,precision,casb_edited,43.139,ros,42.180,0.959,WIN
3,recall,casb_edited,58.447,adasyn,49.017,9.430,WIN


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.339165,0.200000,0.666667,0.117647,0.977778,0.547712,0.554893,0.729412,0.269582,None
1,42,1,svm,smote,1.00,0.699206,0.564103,0.500000,0.647059,0.755556,0.701307,0.567388,0.722876,0.631654,None
2,42,1,svm,bsmote,1.00,0.626224,0.476190,0.400000,0.588235,0.666667,0.627451,0.409792,0.673203,0.551207,None
3,42,1,svm,ros,1.00,0.632456,0.486486,0.450000,0.529412,0.755556,0.642484,0.523233,0.711111,0.559471,None
4,42,1,svm,adasyn,1.00,0.584103,0.428571,0.360000,0.529412,0.644444,0.586928,0.448081,0.640523,0.506337,None
5,42,1,svm,casb,1.00,0.719477,0.594595,0.550000,0.647059,0.800000,0.723529,0.481331,0.704575,0.657036,None
6,42,1,svm,casb_relaxed,1.00,0.623085,0.473684,0.428571,0.529412,0.733333,0.631373,0.547813,0.711111,0.548385,None
7,42,1,svm,casb_edited,1.00,0.674464,0.533333,0.428571,0.705882,0.644444,0.675163,0.531495,0.712418,0.603899,None
8,42,1,svm,casb_ros,1.00,0.650791,0.514286,0.500000,0.529412,0.800000,0.664706,0.468542,0.688889,0.582539,None
9,42,1,knn,original,0.50,0.391077,0.230769,0.333333,0.176471,0.866667,0.521569,0.400699,0.690196,0.310923,None
